# 🎮 Game 2: Limited Visibility

## 🌫️ The New Challenge

**Bad news from Mission Control:** The sensors are damaged! You can no longer see the entire crater surface.

**What you CAN see:** Only a small "window" around your current position (±1.0 units)

**What you CANNOT see:** Everything outside that window is hidden in the fog

### The Crater
Same as before, but now with a different shape:

**L(w) = w² - 4w + 7**

**Optimal position:** w* = 2.0 (but you'll need to find it!)

### Your Mission

Using only the **local information** in your small visibility window, try to navigate to the minimum.

**Key Questions:**
- Can you figure out which direction to move just by looking at the local shape?
- What does a steep slope tell you vs. a gentle slope?
- Can you still find the minimum without seeing the whole picture?

### 💡 What You'll Discover

The **local shape** of the curve contains all the information you need to navigate toward the minimum. This is the foundation of gradient-based optimization!

**Ready? Let's explore! 🚀**

In [1]:
#@title 🔧 Game 2 Setup Code (Limited Visibility) { display-mode: "form" }

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import Button, VBox, HBox, Output, HTML, FloatText
from IPython.display import display, clear_output

class Game2_LimitedVisibility:
    def __init__(self):
        # Loss function (different from Game 1)
        self.L = lambda w: w**2 - 4*w + 7
        self.optimal_w = 2.0
        self.optimal_loss = self.L(self.optimal_w)

        # Visibility window size (REDUCED)
        self.window_size = 0.5  # Much smaller now!

        # Game state
        self.current_w = -1.0  # Start position
        self.history = []
        self.output = Output()

    def create_ui(self):
        """Create the interactive UI"""

        # Position input with smaller step size
        self.position_input = FloatText(
            value=self.current_w,
            description='Move to w:',
            step=0.1,  # ← FIXED: Now increments by 0.1 instead of 1.0
            style={'description_width': '100px'},
            layout={'width': '300px', 'height': '40px'}
        )

        # Buttons
        move_button = Button(
            description='👁️ Look & Move',
            button_style='',
            layout={'width': '140px', 'height': '40px'}
        )
        move_button.style.button_color = '#3b82f6'
        move_button.style.text_color = 'white'
        move_button.style.font_weight = 'bold'
        move_button.on_click(self.move_to_position)

        reset_button = Button(
            description='🔄 Reset',
            button_style='',
            layout={'width': '140px', 'height': '40px'}
        )
        reset_button.style.button_color = '#64748b'
        reset_button.style.text_color = 'white'
        reset_button.style.font_weight = 'bold'
        reset_button.on_click(self.reset)

        reveal_button = Button(
            description='🔦 Reveal All',
            button_style='',
            layout={'width': '140px', 'height': '40px'}
        )
        reveal_button.style.button_color = '#f59e0b'
        reveal_button.style.text_color = 'white'
        reveal_button.style.font_weight = 'bold'
        reveal_button.on_click(self.reveal_all)

        # Info display
        self.info_display = HTML("""
        <div style='background: #1e293b; padding: 15px; border-radius: 8px; border: 1px solid #475569;'>
            <p style='color: #e2e8f0; margin: 5px 0; font-size: 15px;'><strong>Current Position:</strong> w = -1.00</p>
            <p style='color: #e2e8f0; margin: 5px 0; font-size: 15px;'><strong>Visible Range:</strong> [-1.50, -0.50]</p>
            <p style='color: #e2e8f0; margin: 5px 0; font-size: 15px;'><strong>Local Observation:</strong> Slopes downward to the right ↘️</p>
            <p style='color: #e2e8f0; margin: 5px 0; font-size: 15px;'><strong>Moves:</strong> 0</p>
        </div>
        """)

        # Layout
        control_box = HBox([self.position_input, move_button],
                          layout={'margin': '10px 0', 'align_items': 'center'})
        button_box = HBox([reset_button, reveal_button],
                         layout={'margin': '10px 0'})

        ui = VBox([
            self.output,
            control_box,
            button_box,
            self.info_display
        ])

        display(ui)
        self.reset(None)

    def reset(self, b):
        """Reset to starting position"""
        self.current_w = -1.0
        self.history = [self.current_w]
        self.position_input.value = self.current_w
        self.plot_limited_view()
        self.update_info()

    def move_to_position(self, b):
        """Move to the specified position"""
        try:
            new_w = float(self.position_input.value)
        except ValueError:
            with self.output:
                clear_output(wait=True)
                print("⚠️ Please enter a valid number")
            return

        self.current_w = new_w
        self.history.append(new_w)
        self.plot_limited_view()
        self.update_info()

    def reveal_all(self, b):
        """Reveal the full crater (cheat mode)"""
        self.plot_full_view()

    def plot_limited_view(self):
        """Plot only the visible window around current position"""
        with self.output:
            clear_output(wait=True)

            fig, ax = plt.subplots(figsize=(14, 8), facecolor='#0a0e27')
            ax.set_facecolor('#1a1a2e')

            # Define visible range (smaller window)
            w_min = self.current_w - self.window_size
            w_max = self.current_w + self.window_size

            # Visible range ONLY
            visible_w_range = np.linspace(w_min, w_max, 100)
            visible_loss_range = self.L(visible_w_range)

            # Plot ONLY the visible range (NO dotted lines showing hidden areas)
            ax.plot(visible_w_range, visible_loss_range, color='#8b8b8b',
                   linewidth=4, label='Visible Surface', zorder=3)
            ax.fill_between(visible_w_range, visible_loss_range,
                           min(visible_loss_range) - 1,
                           alpha=0.3, color='#3a3a3a', zorder=2)

            # Highlight the visibility window boundaries
            ax.axvline(x=w_min, color='#fbbf24', linestyle='-',
                      linewidth=3, alpha=0.7, label='Visibility Boundary', zorder=4)
            ax.axvline(x=w_max, color='#fbbf24', linestyle='-',
                      linewidth=3, alpha=0.7, zorder=4)

            # Add fog walls (solid blocks)
            fog_height = max(visible_loss_range) + 5
            ax.fill_between([w_min - 10, w_min], [0, 0], [fog_height, fog_height],
                           color='#1a1a2e', alpha=0.95, zorder=10)
            ax.fill_between([w_max, w_max + 10], [0, 0], [fog_height, fog_height],
                           color='#1a1a2e', alpha=0.95, zorder=10)

            # Current position
            ax.scatter([self.current_w], [self.L(self.current_w)],
                      color='#ef4444', s=300, marker='v',
                      edgecolors='#dc2626', linewidths=3, zorder=6,
                      label='Your Position')

            # Add glow around current position
            ax.scatter([self.current_w], [self.L(self.current_w)],
                      color='#fecaca', s=600, alpha=0.3, marker='o', zorder=5)

            # Show ONLY recent path history (last 3-4 positions within window)
            if len(self.history) > 1:
                recent_history = [w for w in self.history[:-1] if w_min <= w <= w_max][-3:]
                for i, prev_w in enumerate(recent_history):
                    alpha = 0.4 + 0.3 * (i / max(len(recent_history), 1))
                    ax.scatter([prev_w], [self.L(prev_w)],
                             color='#fbbf24', s=80, alpha=alpha,
                             edgecolors='#f59e0b', linewidths=1.5, zorder=4)

            # Add "FOG" text on the boundaries
            y_text = (max(visible_loss_range) + min(visible_loss_range)) / 2
            ax.text(w_min - 0.15, y_text, '🌫️\nFOG',
                   fontsize=18, color='#fbbf24', alpha=0.8,
                   ha='right', va='center', weight='bold')
            ax.text(w_max + 0.15, y_text, '🌫️\nFOG',
                   fontsize=18, color='#fbbf24', alpha=0.8,
                   ha='left', va='center', weight='bold')

            # Check if found the optimal (FIXED: check before setting title)
            if abs(self.current_w - self.optimal_w) < 0.05:
                ax.text(0.5, 0.95, '🎯 MINIMUM FOUND!',
                       transform=ax.transAxes,
                       fontsize=18, fontweight='bold', color='#4ade80',
                       ha='center', va='top',
                       bbox=dict(boxstyle='round,pad=1',
                                facecolor='#14532d', alpha=0.95,
                                edgecolor='#22c55e', linewidth=3),
                       zorder=20)

            # Styling
            ax.set_xlabel('Position (w)', fontsize=14, color='#c7d2fe', fontweight='bold')
            ax.set_ylabel('Altitude L(w)', fontsize=14, color='#c7d2fe', fontweight='bold')
            ax.set_title('LIMITED VISIBILITY - Navigate Using Local Information Only',
                        fontsize=16, color='#e0e7ff', fontweight='bold', pad=20)
            ax.grid(True, alpha=0.2, color='#4a5568', linestyle='--')
            ax.legend(fontsize=11, loc='upper right',
                     facecolor='#1a1a2e', edgecolor='#4a4a6a',
                     labelcolor='#e0e7ff', framealpha=0.9)

            # Set tight limits around visible window
            ax.set_xlim(w_min - 0.3, w_max + 0.3)
            ax.set_ylim(min(visible_loss_range) - 1, max(visible_loss_range) + 2)

            for spine in ax.spines.values():
                spine.set_edgecolor('#4a4a6a')
                spine.set_linewidth(2)
            ax.tick_params(colors='#a5b4fc')

            plt.tight_layout()
            plt.show()

    def plot_full_view(self):
        """Reveal the full crater (for comparison/teaching)"""
        with self.output:
            clear_output(wait=True)

            fig, ax = plt.subplots(figsize=(14, 8), facecolor='#0a0e27')
            ax.set_facecolor('#1a1a2e')

            # Full crater
            w_range = np.linspace(-2, 6, 200)
            loss_range = self.L(w_range)

            ax.plot(w_range, loss_range, color='#8b8b8b',
                   linewidth=3.5, label='Full Crater Surface')
            ax.fill_between(w_range, loss_range, -2, alpha=0.3, color='#3a3a3a')

            # Show full path
            if len(self.history) > 1:
                for i, w in enumerate(self.history):
                    alpha = 0.4 + 0.5 * (i / len(self.history))
                    ax.scatter([w], [self.L(w)], color='#fbbf24', s=100,
                             alpha=alpha, edgecolors='#f59e0b', linewidths=2, zorder=4)

                # Connect path
                path_w = self.history
                path_l = [self.L(w) for w in path_w]
                ax.plot(path_w, path_l, '-', color='#fbbf24',
                       linewidth=2, alpha=0.6, zorder=3, label='Your Path')

            # Current position
            ax.scatter([self.current_w], [self.L(self.current_w)],
                      color='#ef4444', s=300, marker='v',
                      edgecolors='#dc2626', linewidths=3, zorder=6,
                      label='Your Position')

            # Optimal position
            ax.scatter([self.optimal_w], [self.optimal_loss],
                      color='#22c55e', s=400, marker='*',
                      edgecolors='#16a34a', linewidths=3, zorder=6,
                      label='Optimal Position')

            # Glow effects
            for size, alpha in [(800, 0.1), (1200, 0.05)]:
                ax.scatter([self.optimal_w], [self.optimal_loss],
                          color='#86efac', s=size, alpha=alpha, marker='o', zorder=5)

            # Styling
            ax.set_xlabel('Position (w)', fontsize=14, color='#c7d2fe', fontweight='bold')
            ax.set_ylabel('Altitude L(w)', fontsize=14, color='#c7d2fe', fontweight='bold')
            ax.set_title('FULL VIEW REVEALED - Your Navigation Path',
                        fontsize=16, color='#e0e7ff', fontweight='bold', pad=20)
            ax.grid(True, alpha=0.15, color='#4a5568')
            ax.legend(fontsize=12, loc='upper right',
                     facecolor='#1a1a2e', edgecolor='#4a4a6a',
                     labelcolor='#e0e7ff', framealpha=0.9)
            ax.set_xlim(-2, 6)
            ax.set_ylim(-2, max(loss_range) + 2)

            for spine in ax.spines.values():
                spine.set_edgecolor('#4a4a6a')
                spine.set_linewidth(2)
            ax.tick_params(colors='#a5b4fc')

            # Success check
            if abs(self.current_w - self.optimal_w) < 0.2:
                ax.text(0.5, 0.92, 'SUCCESS! You navigated to the minimum!',
                       transform=ax.transAxes,
                       fontsize=16, fontweight='bold', color='#4ade80',
                       ha='center', va='top',
                       bbox=dict(boxstyle='round,pad=1',
                                facecolor='#14532d', alpha=0.95,
                                edgecolor='#22c55e', linewidth=3))

            plt.tight_layout()
            plt.show()

    def update_info(self):
        """Update information display with local observations"""
        w_min = self.current_w - self.window_size
        w_max = self.current_w + self.window_size

        # Analyze local slope (smaller sample for more precise measurement)
        left_val = self.L(self.current_w - 0.2)
        right_val = self.L(self.current_w + 0.2)
        current_val = self.L(self.current_w)

        # FIXED: Check if at optimal FIRST
        distance = abs(self.current_w - self.optimal_w)

        if distance < 0.05:  # At the minimum!
            observation = "PERFECTLY FLAT - This is the minimum! 🎯"
            hint = "✓ You found it! The slope is zero here."
            bg_color = '#14532d'
            border_color = '#22c55e'
            status = "🎉 MINIMUM FOUND!"
        else:
            # Determine slope direction
            if right_val < current_val - 0.01:  # Threshold to avoid floating point issues
                observation = "Slopes downward to the RIGHT ↘️"
                hint = "💡 Try moving right (increase w)"
            elif left_val < current_val - 0.01:
                observation = "Slopes downward to the LEFT ↙️"
                hint = "💡 Try moving left (decrease w)"
            elif abs(right_val - left_val) < 0.05:
                observation = "Nearly FLAT - very close to minimum! 🎯"
                hint = "✓ Make small adjustments to find exact minimum"
            else:
                observation = "Curved shape - analyze carefully"
                hint = "Look at which side is lower"

            # Set colors based on distance
            if distance < 0.2:
                bg_color = '#1e40af'
                border_color = '#3b82f6'
                status = "Very close..."
            else:
                bg_color = '#1e293b'
                border_color = '#475569'
                status = "Keep exploring..."

        self.info_display.value = f"""
        <div style='background: {bg_color}; padding: 15px; border-radius: 8px; border: 1px solid {border_color};'>
            <p style='color: #e2e8f0; margin: 5px 0; font-size: 15px;'><strong>Current Position:</strong> w = {self.current_w:.2f}</p>
            <p style='color: #e2e8f0; margin: 5px 0; font-size: 15px;'><strong>Current Altitude:</strong> L(w) = {current_val:.2f}</p>
            <p style='color: #e2e8f0; margin: 5px 0; font-size: 15px;'><strong>Visible Range:</strong> [{w_min:.2f}, {w_max:.2f}]</p>
            <p style='color: #fbbf24; margin: 5px 0; font-size: 15px;'><strong>Local Observation:</strong> {observation}</p>
            <p style='color: #94a3b8; margin: 5px 0; font-size: 14px; font-style: italic;'>{hint}</p>
            <p style='color: #e2e8f0; margin: 5px 0; font-size: 15px;'><strong>Moves:</strong> {len(self.history) - 1} | <strong>Status:</strong> {status}</p>
        </div>
        """

print("✅ Game 2 setup complete! Navigate using only local information.")

✅ Game 2 setup complete! Navigate using only local information.


In [2]:
#@title ▶️ Launch Game 2: Limited Visibility { display-mode: "form" }
#@markdown Navigate to the minimum using only what you can see in your small window!

game2 = Game2_LimitedVisibility()
game2.create_ui()

## 🤔 Reflection: Game 2

Take a moment to think deeply about what you just discovered:

### Question 1: Success with Limited Information
**Were you able to find the minimum using only the small visible window?**

Think about:
- How many moves did it take you?
- Did you need to see the entire crater, or was the local view enough?
- What was your strategy for deciding which direction to move?

### Question 2: The Local Shape is Key
**What information from the visible window helped you navigate?**

When you looked at the small visible section:
- How did you know whether to move left or right?
- What did a **steep slope** tell you vs a **gentle slope**?
- When the curve was nearly **flat**, what did that mean?

### Question 3: From Intuition to Mathematics
**How would you MEASURE the slope you were observing?**

Imagine you wanted to write code to automatically detect "which way is downhill":
- If you sample the loss at two points in your window, how would you calculate the slope between them?
- What information would you need: L(left point), L(right point), and...?

### Question 4: The Key Insight
**Compare your experience across games:**

- **Game 1A** (Manual search): Tedious, uncertain, doesn't scale
- **Game 1B** (Oracle): Perfect convergence, but requires knowing w* (impossible in practice)
- **Game 2** (Local information): Successfully navigated using only the visible slope!

**The breakthrough:** You don't need to know where the optimal point is. You just need to know which direction is downhill from where you currently are!

---

### 💡 What You've Discovered

You've just experienced the core principle of **gradient-based optimization**:

> **The local slope contains all the information needed to navigate toward the minimum.**

- ✅ No need to see the entire landscape
- ✅ No need for an oracle telling you w*
- ✅ Just measure the slope and move downhill

**But here's the question:** How do we measure this slope **mathematically**?